# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`. The metadata provides a comprehensive overview of the dataset, including description, authors, keywords, and more.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata and object
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title: ", metadata.name)
print("Description: ", metadata.description)

# Print additional metadata information
print("Identifier: ", getattr(metadata, 'identifier', None))
print("Published: ", getattr(metadata, 'datePublished', None))
print("Authors: ", [author['@id'] for author in getattr(metadata, 'author', [])])
print("Keywords: ", getattr(metadata, 'keywords', []))

## 2. Data Overview
Review available record sets and their IDs. You can inspect the record set structure and learn field IDs for later extraction. All entities should be referenced by their `@id` fields.

In [ ]:
# List available record sets from the metadata
# The recordSet property contains a list of record set entities with their @id
record_sets = [rs['@id'] for rs in getattr(metadata, 'recordSet', [])]

if not record_sets:
    print("No record sets found in metadata.")
else:
    print("Available Record Sets (@id):")
    for rs in getattr(metadata, 'recordSet', []):
        print(f"  @id: {rs['@id']}")
        # List available fields in each record set
        if 'field' in rs:
            print("    Fields:")
            for field in rs['field']:
                print(f"      @id: {field['@id']}, name: {field.get('name', '')}")

# Example: Print the first few records of a record set if available
if record_sets:
    example_rs = record_sets[0]  # Use the first record set @id
    print(f"\nLoading sample records from record set: {example_rs}")
    for i, x in enumerate(dataset.records(record_set=example_rs)):
        pprint.pprint(x)
        if i >= 2:
            break

## 3. Data Extraction
Load data from the key record sets into Pandas DataFrames for analysis. Use the record set and field `@id`s from the overview above. We'll iterate through all available record sets and extract their records.

In [ ]:
dataframes = {}

if not record_sets:
    print("No record sets to extract.")
else:
    print("Extracting records into dataframes:")
    for rs_id in record_sets:
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"  @id: {rs_id}, shape: {dataframes[rs_id].shape}")

    # Display columns of the first record set DataFrame
    first_rs = record_sets[0]
    print(f"\nColumns in record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    # Show head
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Let's apply common data processing steps such as filtering records based on a numeric field, normalizing values, and grouping records by key attributes. All field references use their `@id`. Modify the field IDs below according to the available fields in your actual record set.

In [ ]:
# Example: use the first record set for EDA
eda_rs_id = record_sets[0] if record_sets else None
# Choose numeric_field_id and group_field_id based on available columns

if eda_rs_id and not dataframes[eda_rs_id].empty:
    df = dataframes[eda_rs_id]
    # Attempt to find numeric fields
    numeric_fields = df.select_dtypes(include=['float', 'int']).columns.tolist()
    group_fields = df.select_dtypes(include=['object']).columns.tolist()
    print(f"Numeric fields: {numeric_fields}")
    print(f"Group fields: {group_fields}")

    # For demonstration, pick the first numeric field
    if numeric_fields:
        numeric_field = numeric_fields[0]
        threshold = df[numeric_field].mean() if pd.notnull(df[numeric_field].mean()) else 10
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold}:")
        print(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Group by a group field if available
        if group_fields:
            group_field = group_fields[0]
            grouped_df = filtered_df.groupby(group_field).mean(numeric_only=True)
            print(f"Grouped data by {group_field}:")
            print(grouped_df.head())
else:
    print("No data available for EDA.")

## 5. Visualization
Visualize data distributions and relationships between fields using matplotlib or seaborn. Modify the field IDs and DataFrame accordingly.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if eda_rs_id and not dataframes[eda_rs_id].empty and numeric_fields:
    df = dataframes[eda_rs_id]
    numeric_field = numeric_fields[0]
    plt.figure(figsize=(8, 6))
    sns.histplot(df[numeric_field], bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()

    if group_fields:
        group_field = group_fields[0]
        plt.figure(figsize=(10, 6))
        sns.boxplot(x=df[group_field], y=df[numeric_field])
        plt.title(f"{numeric_field} grouped by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
This notebook demonstrated how to load and explore the FAIR^2 dataset package using `mlcroissant`, referencing data entities by their `@id`. 

Key steps included:
- Loading the Croissant schema and inspecting metadata
- Listing available record sets and fields by `@id`
- Extracting tabular data into Pandas DataFrames
- Applying basic EDA, including filtering, normalization, and grouping
- Visualizing distributions and relationships

Further analyses can build on this template for deeper investigation of clinical and molecular predictors in second primary colorectal cancer survivors.